In [5]:
import pygad
from scipy.stats import norm
import matplotlib.pyplot as plt
import json
from fitness import *
import numpy as np

In [6]:
with open("../datasets/hard_dataset_3.json", "r") as file:
    data = json.load(file)

# Step 2: Extract information
dataset_id = data["dataset_id"]
num_papers = data["num_papers"]
num_reviewers = data["num_reviewers"]
reviewer_capacity = data["reviewer_capacity"]
min_reviews_per_paper = data["min_reviews_per_paper"]
max_reviews_per_paper = data["max_reviews_per_paper"]
preferences = np.array(data["preferences"])
friendship = np.array(data["friendships"])
authorship = np.array(data["authorship"])

In [7]:
# Dynamic parameters based on stagnant generations
stagnant_generations = 0
max_stagnant_generations = 10
initial_mutation_probability = 0.2
best_solution_fitness = 0

# a chance to repair a constraint
repair_chance = 0.2
# a weight of additional points for new constraint
spread_weight = 0.8

# generation counter
gen_num = 1

def check(ga_instance, children):
    # Repair children
    for idx, child in enumerate(children):
        matr = np.reshape(child, (num_reviewers, num_papers))
        if not is_valid_assignment(matr):
            children[idx] = repair_solution(matr).reshape(-1)
    return children

def check_initial(ga_instance):
    # Repair initial population
    for solution_idx, solution in enumerate(ga_instance.population):
        assignment = solution.reshape((num_reviewers, num_papers))
        if not is_valid_assignment(assignment):
            ga_instance.population[solution_idx] = repair_solution(assignment).reshape(-1)

# Validity checks (constraints)
def is_valid_assignment(assignment):
    # Check paper reviews (min and max reviews per paper)  - Skipped because with our correction algorithm, we are not fixing this constraint 
    # (cannot satisfy all constraints with simple correction ... or?)
    #
    # paper_counts = np.sum(assignment, axis=0)
    # if any(paper_counts < min_reviews_per_paper) or any(paper_counts > max_reviews_per_paper):
    #     return False
    
    # Check reviewer capacities
    reviewer_counts = np.sum(assignment, axis=1)
    if any(reviewer_counts > reviewer_capacity):
        return False
    
    # Check authorship constraints (a reviewer cannot review their own paper)
    if np.any(assignment & authorship):
        return False

    # Check friendship constraints (no two friends should review the same paper)
    for i in range(num_reviewers):
        for j in range(num_reviewers):
            if friendship[i][j] == 1:
                for k in range(num_papers):
                    if assignment[i][k] == 1 and assignment[j][k] == 1:
                        return False

    return True

# Repair function to correct an invalid solution
def repair_solution(assignment):
    global repair_chance

    # Ensure the offspring meet the constraints
    # 1. Reviewer capacity (max reviews per reviewer)
    for i in range(num_reviewers):
        reviewer_count = np.sum(assignment[i])
        if reviewer_count > reviewer_capacity:
            for j in range(num_papers):
                if assignment[i, j] == 1 and reviewer_count > reviewer_capacity and np.random.rand() < repair_chance:
                    assignment[i, j] = 0
                    reviewer_count -= 1
    
    # 2. Paper review requirements (each paper needs between min_reviews_per_paper and max_reviews_per_paper reviews)
    for j in range(num_papers):
        paper_count = np.sum(assignment[:, j])
        
        # Ensure the paper has at least min_reviews_per_paper reviews
        if paper_count < min_reviews_per_paper:
            for i in range(num_reviewers):
                if assignment[i, j] == 0 and np.random.rand() < repair_chance:
                    assignment[i, j] = 1
                    paper_count += 1
                    if paper_count >= min_reviews_per_paper:
                        break
        
        # Remove excess reviewers from the paper if it exceeds max_reviews_per_paper
        if paper_count > max_reviews_per_paper:
            for i in range(num_reviewers):
                if assignment[i, j] == 1 and paper_count > max_reviews_per_paper and np.random.rand() < repair_chance:
                    assignment[i, j] = 0
                    paper_count -= 1
    
    # 3. Friendship constraints (no two friends should review the same paper)
    for i in range(num_reviewers):
        for j in range(i + 1, num_reviewers):
            if friendship[i, j] == 1:
                for k in range(num_papers):
                    if assignment[i, k] == 1 and assignment[j, k] == 1:
                        # Swap one reviewer's assignment to a different paper
                        for l in range(num_papers):
                            if assignment[i, l] == 0 and np.random.rand() < repair_chance:
                                assignment[i, l] = 1
                                assignment[i, k] = 0
                                break
    
    # 4. Authorship constraints (reviewers cannot review their own papers)
    for i in range(num_reviewers):
        for j in range(num_papers):
            if authorship[i, j] == 1 and np.random.rand() < repair_chance:
                assignment[i, j] = 0

    return assignment

def fitness_func(ga_instance, solution, solution_idx):
    global spread_weight
    assignment = solution.reshape(num_reviewers, num_papers)
    
    # Penalty weights
    rlpp = 10
    plpr = 10
    fc = 20
    ac = 50

    # Satisfaction score (primary objective)
    satisfaction = np.sum(assignment * preferences)
    
    # Secondary objective: maximize the spread of preferences across reviewers
    spread_score = 0
    for i in range(num_reviewers):
        # Get the preferences for papers assigned to reviewer i
        assigned_papers = np.where(assignment[i] == 1)[0]
        if len(assigned_papers) > 0:
            reviewer_preferences = preferences[i, assigned_papers]
            # Calculate the variance of preferences
            mean_preference = np.mean(reviewer_preferences)
            variance = np.var(reviewer_preferences)
            spread_score += variance  # Add variance to the spread score
    
    # Constraints penalties (primary objective constraints)
    penalties = 0
    
    # Review limit per paper
    for j in range(num_papers):
        if np.sum(assignment[:, j]) < min_reviews_per_paper or np.sum(assignment[:, j]) > max_reviews_per_paper:
            penalties += rlpp
    
    # Paper limit per reviewer
    for i in range(num_reviewers):
        if np.sum(assignment[i, :]) > reviewer_capacity:
            penalties += plpr
    
    # Friendship constraint
    for j in range(num_papers):
        reviewers = np.where(assignment[:, j] == 1)[0]
        for r1 in reviewers:
            for r2 in reviewers:
                if friendship[r1, r2] == 1:
                    penalties += fc
    
    # Authorship constraint
    if np.any(assignment * authorship):
        penalties += ac
    
    # The final fitness is the primary objective (satisfaction) plus the secondary objective (spread_score)
    # We want to maximize both objectives, so we add the spread score to the satisfaction
    # The spread score can also be scaled if necessary, e.g., by a constant factor
    pen_weight = 1#-spread_weight
    return satisfaction + spread_weight * spread_score - penalties * pen_weight


binary_gene_space = [0,1]
# Define the genetic algorithm using the custom GA class

In [8]:
num_iter = 1
avg = 0
seed = 0
max_sol = 0

for i in range(num_iter):
    seed += 1
    ga_instance = pygad.GA(
        num_generations=100,
        num_parents_mating=20,
        fitness_func=fitness_func,
        sol_per_pop=500,
        num_genes=num_reviewers * num_papers,
        gene_type=int,
        gene_space=binary_gene_space,
        mutation_type="swap", 
        crossover_type="uniform", 
        mutation_probability=0.2,  # Dynamic mutation
        keep_elitism=10,  # Preserve the top 10 solutions
        on_start=check_initial,
        on_mutation=check,
        on_crossover=check,
        random_seed=seed
    )
    ga_instance.run()

    best_solution, best_solution_fitness, _ = ga_instance.best_solution()
    ff = eval_ff(best_solution, preferences, num_reviewers, num_papers, max_reviews_per_paper, min_reviews_per_paper, reviewer_capacity, authorship, friendship)
    print(ff)
    avg += ff
    if (ff > max_sol):
        max_sol = ff

print("avg", avg/20, "max", max_sol)

/home/matjaz/.conda/envs/IS/lib/python3.13/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


248
avg 12.4 max 248


In [9]:
print(eval_ff(best_solution, preferences, num_reviewers, num_papers, max_reviews_per_paper, min_reviews_per_paper, reviewer_capacity, authorship, friendship))

248
